# Extended Peetre benchmark: polynomial, separable, joint, cocktail

This notebook extends the original Peetre notebook. It compares:

- backend='direct' : full Kohn-Nirenberg ground truth
- backend='peetre', joint_backend='direct' : exact Peetre decomposition with direct joint residual
- backend='peetre', joint_backend='lowrank' : Peetre decomposition with low-rank approximate joint residual

The low-rank joint backend approximates only the joint residual as a sum of separable terms:

    p_joint(x, xi) ~= sum_k a_k(x) q_k(xi)

Direct Kohn-Nirenberg application is kept as the reference.

In [ ]:
%matplotlib inline
import warnings
import time
import itertools

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

from psiop import PseudoDifferentialOperator


def relative_error(a, b):
    num = np.linalg.norm(a - b)
    den = np.linalg.norm(b)
    return float(num / den) if den > 1e-16 else float(num)


def make_1d_periodic_grid(N=128, L=8*np.pi):
    x = np.linspace(-L/2, L/2, N, endpoint=False)
    dx = x[1] - x[0]
    kx = 2.0 * np.pi * np.fft.fftfreq(N, d=dx)
    return x, kx


def make_2d_periodic_grid(Nx=32, Ny=32, Lx=4*np.pi, Ly=4*np.pi):
    x = np.linspace(-Lx/2, Lx/2, Nx, endpoint=False)
    y = np.linspace(-Ly/2, Ly/2, Ny, endpoint=False)

    dx = x[1] - x[0]
    dy = y[1] - y[0]

    kx = 2.0 * np.pi * np.fft.fftfreq(Nx, d=dx)
    ky = 2.0 * np.pi * np.fft.fftfreq(Ny, d=dy)

    X, Y = np.meshgrid(x, y, indexing='ij')

    return x, y, kx, ky, X, Y


In [ ]:
def _as_sample_array(val, n):
    arr = np.asarray(val, dtype=np.complex128).reshape(-1)

    if arr.size == 1:
        return np.full(n, arr.item(), dtype=np.complex128)

    if arr.size != n:
        arr = np.broadcast_to(arr, (n,)).astype(np.complex128)

    return arr


def _complex_sym_number(z, digits=6):
    z = complex(z)

    re = float(np.real(z))
    im = float(np.imag(z))

    if abs(re) < 1e-14:
        re = 0.0
    if abs(im) < 1e-14:
        im = 0.0

    if im == 0.0:
        return sp.Float(re, digits)

    return sp.Float(re, digits) + sp.I * sp.Float(im, digits)


def _chebyshev_polynomial(n, z):
    if n == 0:
        return sp.S.One
    if n == 1:
        return z

    t_prev = sp.S.One
    t_curr = z

    for _ in range(2, n + 1):
        t_prev, t_curr = t_curr, sp.expand(2 * z * t_curr - t_prev)

    return t_curr


def lowrank_factorize(
    expr,
    x_syms,
    xi_syms,
    bounds,
    degree=6,
    tol=1e-5,
    num_samples=2000,
    seed=42,
):
    if degree < 1:
        raise ValueError('degree must be >= 1.')

    x_syms = list(x_syms)
    xi_syms = list(xi_syms)
    all_syms = x_syms + xi_syms

    empty_metrics = {
        'n_modes': 0,
        'svd_energy_retained_pct': 100.0,
        'symbol_rel_l2_error': 0.0,
    }

    nodes_1d = [
        np.cos(np.pi * np.arange(degree + 1) / degree)
        for _ in all_syms
    ]

    norm_vars = {}
    phys_from_norm = []

    for s in all_syms:
        lo, hi = bounds[s]
        lo = float(lo)
        hi = float(hi)

        if hi <= lo:
            lo -= 1.0
            hi = lo + 2.0

        norm_vars[s] = (2 * s - (lo + hi)) / (hi - lo)

        phys_from_norm.append(
            lambda z, lo=lo, hi=hi: 0.5 * (lo + hi) + 0.5 * (hi - lo) * z
        )

    grid_coords = [
        phys_from_norm[idx](nodes_1d[idx])
        for idx in range(len(all_syms))
    ]

    mesh = np.meshgrid(*grid_coords, indexing='ij')

    f_grid = sp.lambdify(all_syms, expr, modules='numpy')
    P = np.asarray(f_grid(*mesh), dtype=np.complex128)

    target_shape = mesh[0].shape
    if P.shape != target_shape:
        P = np.broadcast_to(P, target_shape).astype(np.complex128)

    P = P.copy()

    if np.allclose(P, 0.0, atol=1e-14):
        return [], empty_metrics

    vands = [
        np.polynomial.chebyshev.chebvander(nodes_1d[i], degree)
        for i in range(len(all_syms))
    ]

    C = P

    for i, V in enumerate(vands):
        inv_V = np.linalg.inv(V)

        C = np.moveaxis(C, i, 0)
        sh = C.shape
        C = inv_V @ C.reshape(sh[0], -1)
        C = C.reshape(sh)
        C = np.moveaxis(C, 0, i)

    Nx = (degree + 1) ** len(x_syms)
    Nxi = (degree + 1) ** len(xi_syms)

    Cmat = C.reshape((Nx, Nxi))

    U, S, Vt = np.linalg.svd(Cmat, full_matrices=False)

    if S.size == 0 or S[0] == 0:
        return [], empty_metrics

    keep = S > (S[0] * tol)

    if not np.any(keep):
        keep = np.zeros_like(S, dtype=bool)
        keep[0] = True

    energy_den = float(np.sum(S ** 2))
    energy_retained = (
        100.0 * float(np.sum(S[keep] ** 2)) / energy_den
        if energy_den > 0.0 else 100.0
    )

    spatial_multi_indices = list(
        itertools.product(range(degree + 1), repeat=len(x_syms))
    )
    spectral_multi_indices = list(
        itertools.product(range(degree + 1), repeat=len(xi_syms))
    )

    pairs = []

    S_keep = S[keep]
    U_keep = U[:, keep]
    Vt_keep = Vt[keep, :]

    for k in range(len(S_keep)):
        sigma_k = S_keep[k]
        u_k = U_keep[:, k]
        v_k = Vt_keep[k, :]

        a_expr = sp.S.Zero
        for idx, multi_idx in enumerate(spatial_multi_indices):
            coeff = np.sqrt(sigma_k) * u_k[idx]

            if np.abs(coeff) > tol:
                if len(multi_idx) == 0:
                    basis = sp.S.One
                else:
                    basis = sp.Mul(
                        *[
                            _chebyshev_polynomial(deg, norm_vars[x_syms[m]])
                            for m, deg in enumerate(multi_idx)
                        ]
                    )

                a_expr += _complex_sym_number(coeff) * basis

        q_expr = sp.S.Zero
        for idx, multi_idx in enumerate(spectral_multi_indices):
            coeff = np.sqrt(sigma_k) * v_k[idx]

            if np.abs(coeff) > tol:
                if len(multi_idx) == 0:
                    basis = sp.S.One
                else:
                    basis = sp.Mul(
                        *[
                            _chebyshev_polynomial(deg, norm_vars[xi_syms[n]])
                            for n, deg in enumerate(multi_idx)
                        ]
                    )

                q_expr += _complex_sym_number(coeff) * basis

        pairs.append((sp.expand(a_expr), sp.expand(q_expr)))

    if num_samples > 0:
        rng = np.random.default_rng(seed)

        samples = {}
        for s in all_syms:
            lo, hi = bounds[s]
            samples[s] = rng.uniform(float(lo), float(hi), size=num_samples)

        f_mc = sp.lambdify(all_syms, expr, modules='numpy')
        y_true = _as_sample_array(
            f_mc(*[samples[s] for s in all_syms]),
            num_samples,
        )

        y_approx = np.zeros(num_samples, dtype=np.complex128)

        x_pts = [samples[s] for s in x_syms]
        xi_pts = [samples[s] for s in xi_syms]

        for a_k, q_k in pairs:
            fa = sp.lambdify(x_syms, a_k, modules='numpy')
            fq = sp.lambdify(xi_syms, q_k, modules='numpy')

            va = _as_sample_array(fa(*x_pts), num_samples)
            vq = _as_sample_array(fq(*xi_pts), num_samples)

            y_approx += va * vq

        diff = y_true - y_approx
        n_true = np.linalg.norm(y_true)
        n_diff = np.linalg.norm(diff)

        rel_err = float(n_diff / n_true) if n_true > 0 else float(n_diff)
    else:
        rel_err = np.nan

    metrics = {
        'n_modes': len(pairs),
        'svd_energy_retained_pct': energy_retained,
        'symbol_rel_l2_error': rel_err,
        'singular_values': S_keep,
    }

    return pairs, metrics


In [ ]:
def _extract_variables(expr, vars_x, dim):
    free = getattr(expr, 'free_symbols', set())

    x_syms = []
    for v in vars_x:
        s = next((fs for fs in free if fs.name == v.name), None)
        x_syms.append(s if s is not None else v)

    freq_names = ['xi'] if dim == 1 else ['xi', 'eta']

    xi_syms = []
    for name in freq_names:
        s = next((fs for fs in free if fs.name == name), None)
        if s is None:
            s = sp.symbols(name, real=True)
        xi_syms.append(s)

    return x_syms, xi_syms


def _remap_bounds(bounds, syms):
    out = {}

    for s in syms:
        if s in bounds:
            out[s] = bounds[s]
            continue

        matched_key = None
        for k in bounds.keys():
            k_name = getattr(k, 'name', str(k))
            s_name = getattr(s, 'name', str(s))

            if k_name == s_name:
                matched_key = k
                break

        if matched_key is None:
            raise ValueError(f'No bound provided for symbol {s}.')

        out[s] = bounds[matched_key]

    return out


def _infer_bounds_for_symbols(x_syms, xi_syms, x_grid, kx, y_grid=None, ky=None):
    def _bounds(arr):
        arr = np.asarray(arr)

        if arr.size == 0:
            raise ValueError('Empty grid encountered while inferring bounds.')

        lo = float(np.min(arr))
        hi = float(np.max(arr))

        if hi <= lo:
            lo -= 1.0
            hi += 1.0

        return lo, hi

    bounds = {}

    if len(x_syms) >= 1:
        bounds[x_syms[0]] = _bounds(x_grid)

    if len(x_syms) >= 2:
        if y_grid is None:
            raise ValueError('y_grid is required for 2D bounds.')
        bounds[x_syms[1]] = _bounds(y_grid)

    if len(xi_syms) >= 1:
        bounds[xi_syms[0]] = _bounds(kx)

    if len(xi_syms) >= 2:
        if ky is None:
            raise ValueError('ky is required for 2D bounds.')
        bounds[xi_syms[1]] = _bounds(ky)

    return bounds


def _get_effective_decomposition(
    op,
    weyl_order=4,
    use_cache=True,
    separable_local=False,
):
    if op.quantization == 'weyl':
        effective_symbol = op.weyl_to_kn_symbol(order=weyl_order)

        effective_op = PseudoDifferentialOperator(
            effective_symbol,
            op.vars_x,
            mode='symbol',
            quantization='kohn-nirenberg',
        )

        deco = effective_op.peetre_decomposition(
            use_cache=use_cache,
            separable_local=separable_local,
        )

        return deco, 'kohn-nirenberg', effective_symbol

    deco = op.peetre_decomposition(
        use_cache=use_cache,
        separable_local=separable_local,
    )

    return deco, op.quantization, op.symbol


def _apply_separable_pair_generic(
    op,
    a,
    q,
    x_syms,
    quantization,
    u,
    x_grid,
    kx,
    y_grid,
    ky,
    common,
):
    common = dict(common or {})
    common.pop('backend', None)

    op_q = PseudoDifferentialOperator(
        q,
        op.vars_x,
        mode='symbol',
        quantization=quantization,
    )

    v = op_q.apply(
        u,
        x_grid,
        kx,
        y_grid=y_grid,
        ky=ky,
        backend='direct',
        **common,
    )

    try:
        a_func = sp.lambdify(tuple(x_syms), a, modules='numpy')

        if len(x_syms) == 1:
            a_vals = a_func(x_grid)
        elif len(x_syms) == 2:
            if y_grid is None:
                raise ValueError('y_grid is required for 2D spatial amplitudes.')
            X, Y = np.meshgrid(x_grid, y_grid, indexing='ij')
            a_vals = a_func(X, Y)
        else:
            raise NotImplementedError('Only 1D and 2D separable terms are supported.')

        a_vals = np.asarray(a_vals, dtype=np.complex128)

        if a_vals.shape != v.shape:
            a_vals = np.broadcast_to(a_vals, v.shape).astype(np.complex128)

        return a_vals * v

    except Exception as exc:
        warnings.warn(
            'Could not lambdify a separable spatial amplitude. '
            'Falling back to full symbol application.'
        )

        op_full = PseudoDifferentialOperator(
            a * q,
            op.vars_x,
            mode='symbol',
            quantization=quantization,
        )

        return op_full.apply(
            u,
            x_grid,
            kx,
            y_grid=y_grid,
            ky=ky,
            backend='direct',
            **common,
        )


def apply_backend(
    op,
    u,
    x_grid,
    kx,
    backend,
    joint_backend='direct',
    common=None,
    y_grid=None,
    ky=None,
    decomposition=None,
    effective_quantization=None,
    lowrank_options=None,
):
    common = dict(common or {})
    common.pop('backend', None)

    lowrank_options = dict(lowrank_options or {})

    weyl_order = common.get(
        'weyl_order',
        lowrank_options.get('weyl_order', 4),
    )

    if backend == 'direct':
        v = op.apply(
            u,
            x_grid,
            kx,
            y_grid=y_grid,
            ky=ky,
            backend='direct',
            **common,
        )
        return v, None

    if backend != 'peetre':
        raise ValueError('backend must be direct or peetre.')

    if decomposition is None:
        decomposition, effective_quantization, _ = _get_effective_decomposition(
            op,
            weyl_order=weyl_order,
        )
    elif effective_quantization is None:
        effective_quantization = (
            'kohn-nirenberg'
            if op.quantization == 'weyl'
            else op.quantization
        )

    if joint_backend == 'direct':
        v = op.apply_peetre(
            u,
            x_grid,
            kx,
            y_grid=y_grid,
            ky=ky,
            apply_joint=True,
            decomposition=decomposition,
            **common,
        )
        return v, None

    if joint_backend == 'lowrank':
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')

            v = op.apply_peetre(
                u,
                x_grid,
                kx,
                y_grid=y_grid,
                ky=ky,
                apply_joint=False,
                decomposition=decomposition,
                **common,
            )

        joint_symbol = decomposition.get('joint_symbol', 0)

        if op._peetre_is_zero(joint_symbol):
            metrics = {
                'n_modes': 0,
                'svd_energy_retained_pct': 100.0,
                'symbol_rel_l2_error': 0.0,
                'fallback': None,
            }
            return v, metrics

        try:
            x_syms, xi_syms = _extract_variables(
                joint_symbol,
                op.vars_x,
                op.dim,
            )

            bounds = lowrank_options.get('joint_bounds', None)

            if bounds is None:
                bounds = _infer_bounds_for_symbols(
                    x_syms,
                    xi_syms,
                    x_grid,
                    kx,
                    y_grid=y_grid,
                    ky=ky,
                )
            else:
                bounds = _remap_bounds(bounds, x_syms + xi_syms)

            pairs, metrics = lowrank_factorize(
                joint_symbol,
                x_syms,
                xi_syms,
                bounds,
                degree=lowrank_options.get('joint_degree', 6),
                tol=lowrank_options.get('joint_tol', 1e-5),
                num_samples=lowrank_options.get('joint_num_samples', 2000),
                seed=lowrank_options.get('joint_seed', 42),
            )

            metrics['fallback'] = None

            for a_k, q_k in pairs:
                v = v + _apply_separable_pair_generic(
                    op,
                    a_k,
                    q_k,
                    x_syms,
                    effective_quantization,
                    u,
                    x_grid,
                    kx,
                    y_grid,
                    ky,
                    common,
                )

            return v, metrics

        except Exception as exc:
            warnings.warn(
                'Low-rank joint decomposition failed. '
                'Falling back to exact direct Peetre joint application.'
            )

            v = op.apply_peetre(
                u,
                x_grid,
                kx,
                y_grid=y_grid,
                ky=ky,
                apply_joint=True,
                decomposition=decomposition,
                **common,
            )

            return v, {'fallback': str(exc)}

    raise ValueError('joint_backend must be direct or lowrank.')


In [ ]:
def run_symbol_case(
    name,
    expr,
    vars_x,
    x_grid,
    kx,
    u,
    common,
    lowrank_options=None,
    y_grid=None,
    ky=None,
):
    print('=' * 100)
    print('Case: ' + name)
    print('=' * 100)

    op = PseudoDifferentialOperator(
        expr=expr,
        vars_x=vars_x,
        mode='symbol',
    )

    print('--- Peetre decomposition ---')
    op.print_peetre_decomposition()

    weyl_order = common.get('weyl_order', 4)

    deco, effective_quantization, effective_symbol = _get_effective_decomposition(
        op,
        weyl_order=weyl_order,
    )

    reconstructed = (
        deco['local_symbol']
        + deco['separable_symbol']
        + deco['joint_symbol']
    )

    reconstruction_error = sp.simplify(sp.expand(reconstructed - effective_symbol))

    print('--- Symbolic reconstruction check ---')
    print('Reconstruction error:')
    print(reconstruction_error)

    try:
        ok = (reconstruction_error == 0) or bool(reconstruction_error.equals(0))
    except Exception:
        ok = False

    if not ok:
        print('WARNING: symbolic reconstruction could not be certified as zero.')

    configs = [
        ('direct', 'direct'),
        ('direct', 'lowrank'),
        ('peetre', 'direct'),
        ('peetre', 'lowrank'),
    ]

    v_direct = None
    lowrank_metrics = None
    table = []

    print('--- Numerical application ---')

    for backend, joint_backend in configs:
        if backend == 'direct' and v_direct is not None:
            v = v_direct
            elapsed = 0.0
            metrics = None
        else:
            t0 = time.perf_counter()

            v, metrics = apply_backend(
                op,
                u,
                x_grid,
                kx,
                backend=backend,
                joint_backend=joint_backend,
                common=common,
                y_grid=y_grid,
                ky=ky,
                decomposition=deco,
                effective_quantization=effective_quantization,
                lowrank_options=lowrank_options,
            )

            elapsed = time.perf_counter() - t0

            if backend == 'direct':
                v_direct = v

            if backend == 'peetre' and joint_backend == 'lowrank':
                lowrank_metrics = metrics

        err = relative_error(v, v_direct)

        table.append(
            (
                backend,
                joint_backend,
                elapsed,
                float(np.linalg.norm(v)),
                err,
            )
        )

    print()
    print('backend    | joint_backend  | time (s)     | norm             | rel error vs direct')
    print('-' * 90)

    for backend, joint_backend, elapsed, norm_val, err in table:
        note = ''
        if backend == 'direct' and joint_backend == 'lowrank':
            note = '  (same as direct)'

        print(f'{backend:10} | {joint_backend:14} | {elapsed:12.6f} | {norm_val:16.8e} | {err:22.12e}{note}')

    if lowrank_metrics is not None:
        print('--- Low-rank joint diagnostics ---')

        for key in [
            'n_modes',
            'svd_energy_retained_pct',
            'symbol_rel_l2_error',
            'fallback',
        ]:
            if key in lowrank_metrics:
                print(key + ':', lowrank_metrics[key])

        if 'singular_values' in lowrank_metrics:
            svals = np.asarray(lowrank_metrics['singular_values'])
            if svals.size > 0:
                print('singular values:', svals)

    print()


In [ ]:
x, xi = sp.symbols('x xi', real=True)

L1 = 8.0 * np.pi
x_grid_1d, kx_1d = make_1d_periodic_grid(N=128, L=L1)

u1 = np.exp(-np.cos(2.0 * np.pi * x_grid_1d / L1)) * np.cos(2.0 * x_grid_1d)

common_1d = dict(
    boundary_condition='periodic',
    freq_window='gaussian',
    clamp=1e12,
    space_window=False,
    weyl_order=4,
)

lowrank_options_1d = dict(
    joint_degree=8,
    joint_tol=1e-6,
    joint_num_samples=3000,
    joint_seed=42,
)

cases_1d = {
    '1D polynomial / local': (
        (1 + sp.cos(x)) * xi**2
        + sp.sin(x) * xi
        + sp.cos(2*x)
    ),

    '1D separable nonlocal': (
        sp.exp(sp.cos(x)) * sp.sqrt(1 + xi**2)
    ),

    '1D joint': (
        sp.sqrt(
            1
            + sp.Rational(1, 5) * sp.cos(x)
            + (xi / 5)**2
        )
    ),

    '1D cocktail: local + separable + joint': (
        (1 + sp.cos(x)) * xi**2
        + sp.sin(x) * xi
        + sp.exp(sp.cos(x)) * sp.sqrt(1 + xi**2)
        + sp.Rational(1, 10) * sp.sqrt(
            1
            + sp.Rational(1, 5) * sp.cos(x)
            + (xi / 5)**2
        )
    ),
}

for name, expr in cases_1d.items():
    run_symbol_case(
        name=name,
        expr=expr,
        vars_x=[x],
        x_grid=x_grid_1d,
        kx=kx_1d,
        u=u1,
        common=common_1d,
        lowrank_options=lowrank_options_1d,
    )


In [ ]:
x, y, xi, eta = sp.symbols('x y xi eta', real=True)

Lx = 4.0 * np.pi
Ly = 4.0 * np.pi

x_grid_2d, y_grid_2d, kx_2d, ky_2d, X, Y = make_2d_periodic_grid(
    Nx=32,
    Ny=32,
    Lx=Lx,
    Ly=Ly,
)

u2 = (
    np.exp(
        -np.cos(2.0 * np.pi * X / Lx)
        -np.cos(2.0 * np.pi * Y / Ly)
    )
    * np.cos(2.0 * X)
    * np.sin(Y)
)

common_2d = dict(
    boundary_condition='periodic',
    freq_window='gaussian',
    clamp=1e12,
    space_window=False,
    weyl_order=4,
)

lowrank_options_2d = dict(
    joint_degree=6,
    joint_tol=1e-5,
    joint_num_samples=2000,
    joint_seed=42,
)

cases_2d = {
    '2D polynomial / local': (
        (1 + sp.cos(x) * sp.cos(y)) * (xi**2 + eta**2)
        + sp.sin(x) * sp.sin(y) * xi * eta
        + sp.cos(x + y)
    ),

    '2D separable nonlocal': (
        sp.exp(sp.cos(x) + sp.cos(y))
        * sp.sqrt(1 + xi**2 + eta**2)
    ),

    '2D joint': (
        sp.sqrt(
            1
            + sp.Rational(1, 5) * sp.cos(x) * sp.sin(y)
            + (xi / 4)**2
            + (eta / 4)**2
        )
    ),

    '2D cocktail: local + separable + joint': (
        (1 + sp.cos(x) * sp.cos(y)) * (xi**2 + eta**2)
        + sp.sin(x) * sp.sin(y) * xi * eta
        + sp.exp(sp.cos(x) + sp.cos(y)) * sp.sqrt(1 + xi**2 + eta**2)
        + sp.Rational(1, 10) * sp.sqrt(
            1
            + sp.Rational(1, 5) * sp.cos(x) * sp.sin(y)
            + (xi / 4)**2
            + (eta / 4)**2
        )
    ),
}

for name, expr in cases_2d.items():
    run_symbol_case(
        name=name,
        expr=expr,
        vars_x=[x, y],
        x_grid=x_grid_2d,
        kx=kx_2d,
        u=u2,
        common=common_2d,
        lowrank_options=lowrank_options_2d,
        y_grid=y_grid_2d,
        ky=ky_2d,
    )


## Interpretation

- backend='direct' is the full Kohn-Nirenberg reference.
- backend='peetre', joint_backend='direct' applies the Peetre decomposition and then applies the joint residual with the direct full Kohn-Nirenberg path.
- backend='peetre', joint_backend='lowrank' applies the local and separable parts exactly, then approximates the joint residual by a low-rank Chebyshev/SVD sum of separable terms.

For strict exactness checks, use freq_window=None and clamp=np.inf if the symbol is numerically stable. In this notebook, freq_window='gaussian' and clamp=1e12 are used for stability.